In [3]:
import os

# Define the directory where your images are stored
directory = '/Users/saidev/Downloads/pure_2000/'

# List all files in the directory
files = os.listdir(directory)

# Sort the files to maintain the order
files.sort()

# Define the starting number
start_number = 8001

# Loop through all files and rename them
for index, filename in enumerate(files):
    # Define the new file name starting from 10000
    new_filename = f'seaweed_{start_number + index:05}.jpg'  # Adjust the file extension if needed
    # Construct the full path for the old and new file names
    old_file = os.path.join(directory, filename)
    new_file = os.path.join(directory, new_filename)
    # Rename the file
    os.rename(old_file, new_file)

print("Renaming complete!")


Images have been successfully divided into two folders.


In [49]:
import random
from PIL import Image
import os
import logging

# Disable logging
logging.disable(logging.CRITICAL)

# Define input and output directories with custom parameters
folders = {
    "/Users/saidev/Downloads/kim/defect/other/": {
        "output_dir": "/Users/saidev/Downloads/kim_project_tile/tile/other/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 10,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/feather/": {
        "output_dir": "/Users/saidev/Downloads/kim_project_tile/tile/feather/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 10,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/fish-and-by-products/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/fish-and-by-products/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 10,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/seashell/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/seashell/",
        "min_tile_width": 15,
        "max_tile_width": 15,
        "min_tile_height": 15,
        "max_tile_height": 15,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/shrimp/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/shrimp/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 9,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/string/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/string/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 9,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/rope/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/rope/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 6,
        "max_tile_height": 9,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/net/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/net/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 9,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/straw/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/straw/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 10,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
    "/Users/saidev/Downloads/kim/defect/wood-chips/": {
        "output_dir": "/Users/saidev/Downloads/macd/tile/wood-chips/",
        "min_tile_width": 15,
        "max_tile_width": 25,
        "min_tile_height": 5,
        "max_tile_height": 10,
        "tiles_per_image": 40,
        "custom_dimensions": {},
        "custom_size_checks": {}
    },
}

def get_random_tile_size(min_width, max_width, min_height, max_height):
    """Generate a random tile size within the given width and height bounds."""
    return random.randint(min_width, max_width), random.randint(min_height, max_height)

def get_random_position(img_width, img_height, tile_width, tile_height):
    """Generate a random position for a tile within the bounds of the image."""
    left = random.randint(0, img_width - tile_width)
    upper = random.randint(0, img_height - tile_height)
    return left, upper, left + tile_width, upper + tile_height

def calculate_color_percentage(tile_image):
    """Calculate the percentage of non-white (or non-transparent) pixels in the image."""
    total_pixels = tile_image.width * tile_image.height
    colored_pixels = 0
    
    for pixel in tile_image.getdata():
        if isinstance(pixel, tuple):  # For RGB/RGBA images
            if len(pixel) == 4:  # RGBA image
                _, _, _, a = pixel
                if a > 0:  # Not fully transparent
                    colored_pixels += 1
            else:  # RGB image
                if not all(channel == 255 for channel in pixel):  # Not a white pixel
                    colored_pixels += 1
        else:  # Grayscale image
            if pixel != 255:  # Not a white pixel
                colored_pixels += 1
    
    return colored_pixels / total_pixels

def filter_and_save_tiles(output_dir, tile_images, color_range=(0.4, 0.75)):
    """Filter tiles by color percentage and save those within the specified range to a new folder."""
    filtered_tiles_folder = os.path.join(output_dir, "filtered_tiles")
    os.makedirs(filtered_tiles_folder, exist_ok=True)
    
    for tile_image_path in tile_images:
        try:
            tile_image = Image.open(tile_image_path)
            color_percentage = calculate_color_percentage(tile_image)
            
            if color_range[0] <= color_percentage <= color_range[1]:
                final_tile_path = os.path.join(filtered_tiles_folder, os.path.basename(tile_image_path))
                tile_image.save(final_tile_path)
        except Exception as e:
            pass  # Silently ignore errors

def process_image(image_path, output_dir, min_tile_width, max_tile_width, min_tile_height, max_tile_height, tiles_per_image, custom_dimensions, custom_size_checks):
    """Process a single image by creating random tiles and filtering them based on color percentage."""
    try:
        img = Image.open(image_path)
        original_image_name = os.path.splitext(os.path.basename(image_path))[0]
        custom_size = custom_dimensions.get(original_image_name, None)
        custom_size_check = custom_size_checks.get(original_image_name, (1 * 1, 10 * 1024))
        
        img_width, img_height = img.size
        tile_counter = 1
        created_tiles = []
        
        while tile_counter <= tiles_per_image:
            tile_width, tile_height = custom_size if custom_size else get_random_tile_size(min_tile_width, max_tile_width, min_tile_height, max_tile_height)
            left, upper, right, lower = get_random_position(img_width, img_height, tile_width, tile_height)
            
            tile = img.crop((left, upper, right, lower))
            temp_tile_path = os.path.join(output_dir, f"temp_tile_{tile_counter}_{original_image_name}.png")
            tile.save(temp_tile_path)
            
            tile_size = os.path.getsize(temp_tile_path)
            min_size, max_size = custom_size_check
            if min_size <= tile_size <= max_size:
                final_tile_path = os.path.join(output_dir, f"{tile_counter}_{original_image_name}.png")
                os.rename(temp_tile_path, final_tile_path)
                created_tiles.append(final_tile_path)
                tile_counter += 1
            else:
                os.remove(temp_tile_path)

        # Filter tiles after creating them
        filter_and_save_tiles(output_dir, created_tiles)
    
    except Exception as e:
        pass  # Silently ignore errors

def main():
    """Main function to process all images in the defined folders."""
    for input_dir, params in folders.items():
        output_dir = params["output_dir"]
        os.makedirs(output_dir, exist_ok=True)
        
        for file_name in os.listdir(input_dir):
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                image_path = os.path.join(input_dir, file_name)
                # Exclude 'output_dir' from params before unpacking
                process_image(image_path, output_dir, 
                              min_tile_width=params["min_tile_width"], 
                              max_tile_width=params["max_tile_width"], 
                              min_tile_height=params["min_tile_height"], 
                              max_tile_height=params["max_tile_height"], 
                              tiles_per_image=params["tiles_per_image"], 
                              custom_dimensions=params["custom_dimensions"], 
                              custom_size_checks=params["custom_size_checks"])

if __name__ == "__main__":
    main()


In [85]:
import json
from PIL import Image
import os
import random
import pandas as pd
from colorthief import ColorThief
import cv2
import numpy as np

# Constants 
IMAGE_SIZE = (512, 512)
OUTPUT_CSV_1 = "kim_1000.csv"
OUTPUT_CSV_2 = "kim_750.csv"
OUTPUT_EXCEL_1 = "kim_1000.xlsx"
OUTPUT_EXCEL_2 = "kim_750.xlsx"
OUTPUT_BBOX_CSV_1 = "kim_bbox_1000.csv"  # New CSV for bounding box information for the first folder
OUTPUT_BBOX_CSV_2 = "kim_bbox_750.csv"   # New CSV for bounding box information for the second folder

# Define paths to your directories
dataset_folder = '/Users/saidev/Downloads/newnew/'
plastic_image_folders = [
    '/Users/saidev/Downloads/contest/tile/all/',
]
output_folder_1 = '/Users/saidev/Downloads/madara/1000/'
output_folder_2 = '/Users/saidev/Downloads/madara/750/'
json_output_folder_1 = '/Users/saidev/Downloads/madara/Json_1000/'
json_output_folder_2 = '/Users/saidev/Downloads/madara/Json_750/'

# Create the output directories if they don't exist
os.makedirs(output_folder_1, exist_ok=True)
os.makedirs(output_folder_2, exist_ok=True)
os.makedirs(json_output_folder_1, exist_ok=True)
os.makedirs(json_output_folder_2, exist_ok=True)

def is_image_file(filename):
    valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')
    return filename.lower().endswith(valid_extensions)

def create_gradient_mask(image_size):
    """Create a gradient mask to apply transparency to the edges of an image."""
    width, height = image_size
    gradient = np.zeros((height, width), dtype=np.uint8)

    for y in range(height):
        for x in range(width):
            distance = min(x, y, width - x - 1, height - y - 1)
            gradient[y, x] = int(255 * (distance / min(width, height)))

    return gradient

def apply_opacity(image, alpha):
    """Apply opacity to an image."""
    alpha_channel = Image.new('L', image.size, color=int(255 * alpha))
    return Image.merge('RGBA', (*image.split()[:3], alpha_channel))

def apply_gradient_transparency(image, mask):
    """Apply a gradient transparency to an image using the provided mask."""
    alpha_channel = np.array(image.split()[-1])
    new_alpha = cv2.addWeighted(alpha_channel, 1, mask, 1, 0)
    return Image.merge('RGBA', (*image.split()[:3], Image.fromarray(new_alpha)))

def random_transformations(image):
    """Apply random transformations to an image."""
    # Rotate the image by a random angle
    angle = random.choice([0, 90, 180, 270])
    image = image.rotate(angle)

    # Apply a horizontal flip with a 50% probability
    if random.random() < 0.5:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)

    return image

def process_image(dataset_image, plastic_image, dataset_image_name, plastic_image_name):
    """Process the dataset image and overlay a randomly transformed plastic image."""

    # Apply random transformations to the plastic image
    plastic_image = random_transformations(plastic_image)

    # Get dataset image and plastic image dimensions
    dataset_width, dataset_height = dataset_image.size
    plastic_width, plastic_height = plastic_image.size

    # Ensure the plastic image fits within a 30-pixel margin
    max_x = dataset_width - plastic_width - 30
    max_y = dataset_height - plastic_height - 30

    # Ensure the minimum position is at least 30 pixels from the edge
    random_x = random.randint(30, max_x)
    random_y = random.randint(30, max_y)

    # Create gradient mask and apply it to the plastic image
    mask = create_gradient_mask(plastic_image.size)
    plastic_image = apply_gradient_transparency(plastic_image, mask)

    # Paste the plastic image onto the dataset image
    dataset_image.paste(plastic_image, (random_x, random_y), plastic_image)

    # Save the output image
    output_image_name = f"{os.path.splitext(dataset_image_name)[0]}_{os.path.splitext(plastic_image_name)[0]}.png"
    return dataset_image, output_image_name, random_x, random_y, plastic_image.width, plastic_image.height


def extract_colors(image_path):
    """Extract the top 5 colors from an image."""
    color_thief = ColorThief(image_path)
    palette = color_thief.get_palette(color_count=6)  # Get the top 6 colors

    # Convert palette to a list of hex colors
    hex_palette = ['#%02x%02x%02x' % color for color in palette]
    return hex_palette[:5]  # Return only the top 5 colors

def get_image_class_and_defect(image_name):
    """Extract the class and defect object from the image name."""
    parts = image_name.split('_')
    if len(parts) >= 3:
        image_class = parts[-1].split('.')[0]  # Remove file extension from the last part
        defect_obj = parts[-3]
        return image_class, defect_obj
    return "unknown", "unknown"


def translate_term(term, translations):
    """Translate a term using a predefined translation dictionary."""
    return translations.get(term.lower(), term)

def save_json(json_path, data):
    """Save data to a JSON file."""
    with open(json_path, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, ensure_ascii=False, indent=4)

def main():
    translations = {
        "string": "스트링",
        "aqua": "수산",
        "floating": "부유물",
        "etc": "기타",
        "net": "그물", "rope": "로프", "straw": "지푸라기", "seashell": "조개껍질", "shrimp": "새우",
        "other": "기타 갑각류", "fish-and-by-products": "생선 및 부산물", "wood-chips":"나무조각",
        "feather":"깃털", "microplastic":"미세플라스틱", "plastic bag":"비닐봉지", "floating objects":"기타 부유물",
        "animal bones":"동물 뼈", "bug(insect)":"벌레" 
    }

    dataset_images = [f for f in os.listdir(dataset_folder) if is_image_file(f)]
    
    plastic_images = []
    for folder in plastic_image_folders:
        plastic_images.extend([os.path.join(folder, f) for f in os.listdir(folder) if is_image_file(f)])

    coordinates_log_1 = []
    coordinates_log_2 = []
    bbox_log_1 = []
    bbox_log_2 = []

    image_count = 0

    for dataset_image_name in dataset_images:
        dataset_image_path = os.path.join(dataset_folder, dataset_image_name)
        try:
            dataset_image = Image.open(dataset_image_path).convert("RGBA")
        except Exception as e:
            print(f"Error opening dataset image {dataset_image_name}: {e}")
            continue

        # Randomly select a plastic image for each dataset image
        plastic_image_path = random.choice(plastic_images)
        plastic_image_name = os.path.basename(plastic_image_path)
        try:
            plastic_image = Image.open(plastic_image_path).convert("RGBA")
        except Exception as e:
            print(f"Error opening plastic image {plastic_image_name}: {e}")
            continue

        new_image, output_image_name, random_x, random_y, box_width, box_height = process_image(
            dataset_image, plastic_image, dataset_image_name, plastic_image_name)

        if image_count < 1000:
            specific_output_path = os.path.join(output_folder_1, output_image_name)
            specific_json_output_path = os.path.join(json_output_folder_1, f"{os.path.splitext(output_image_name)[0]}.json")

            new_image.save(specific_output_path, "PNG")

            colors = extract_colors(specific_output_path)

            image_class, defect_obj = get_image_class_and_defect(output_image_name)
            translated_class = translate_term(image_class, translations)
            translated_defect = translate_term(defect_obj, translations)

            log_entry = {
                "image_name": output_image_name,
                "이물질_분류": translated_class,
                "defect_class": image_class,
                "이물질_명칭": translated_defect,
                "defect_obj": defect_obj,
                "coordinates_x": random_x,
                "coordinates_y": random_y,
                "box_width": box_width,
                "box_height": box_height,
                "color_1": colors[0],
                "color_2": colors[1],
                "color_3": colors[2],
                "color_4": colors[3],
                "color_5": colors[4],
            }

            coordinates_log_1.append(log_entry)

            bbox_entry = {
                "image_name": output_image_name,
                "top_x": random_x,
                "top_y": random_y,
                "bottom_x": random_x + box_width,
                "bottom_y": random_y + box_height,
                "class_name": image_class
            }
            bbox_log_1.append(bbox_entry)

            json_log_entry = {
                "image_name": output_image_name,
                "defect_class": image_class,
                "top_x": random_x,
                "top_y": random_y,
                "bot_x": random_x + box_width,
                "bot_y": random_y + box_height,
            }
            save_json(specific_json_output_path, json_log_entry)

        else:
            specific_output_path = os.path.join(output_folder_2, output_image_name)
            specific_json_output_path = os.path.join(json_output_folder_2, f"{os.path.splitext(output_image_name)[0]}.json")

            new_image.save(specific_output_path, "PNG")

            colors = extract_colors(specific_output_path)

            image_class, defect_obj = get_image_class_and_defect(output_image_name)
            translated_class = translate_term(image_class, translations)
            translated_defect = translate_term(defect_obj, translations)

            log_entry = {
                "image_name": output_image_name,
                "이물질_분류": translated_class,
                "defect_class": image_class,
                "이물질_명칭": translated_defect,
                "defect_obj": defect_obj,
                "coordinates_x": random_x,
                "coordinates_y": random_y,
                "box_width": box_width,
                "box_height": box_height,
                "color_1": colors[0],
                "color_2": colors[1],
                "color_3": colors[2],
                "color_4": colors[3],
                "color_5": colors[4],
            }

            coordinates_log_2.append(log_entry)

            bbox_entry = {
                "image_name": output_image_name,
                "top_x": random_x,
                "top_y": random_y,
                "bottom_x": random_x + box_width,
                "bottom_y": random_y + box_height,
                "class_name": image_class
            }
            bbox_log_2.append(bbox_entry)

            json_log_entry = {
                "image_name": output_image_name,
                "defect_class": image_class,
                "top_x": random_x,
                "top_y": random_y,
                "bot_x": random_x + box_width,
                "bot_y": random_y + box_height,
            }
            save_json(specific_json_output_path, json_log_entry)

        image_count += 1  # Increment the image counter

    # Save logs for output_folder_1
    coordinates_df_1 = pd.DataFrame(coordinates_log_1)
    coordinates_df_1.to_csv(os.path.join(output_folder_1, OUTPUT_CSV_1), index=False)
    coordinates_df_1.to_excel(os.path.join(output_folder_1, OUTPUT_EXCEL_1), index=False)

    bbox_df_1 = pd.DataFrame(bbox_log_1)
    bbox_df_1.to_csv(os.path.join(output_folder_1, OUTPUT_BBOX_CSV_1), index=False)

    # Save logs for output_folder_2
    coordinates_df_2 = pd.DataFrame(coordinates_log_2)
    coordinates_df_2.to_csv(os.path.join(output_folder_2, OUTPUT_CSV_2), index=False)
    coordinates_df_2.to_excel(os.path.join(output_folder_2, OUTPUT_EXCEL_2), index=False)

    bbox_df_2 = pd.DataFrame(bbox_log_2)
    bbox_df_2.to_csv(os.path.join(output_folder_2, OUTPUT_BBOX_CSV_2), index=False)

    print("Successfully created new defected images.")
    print(f"Logs saved for output_folder_1 to {os.path.join(output_folder_1, OUTPUT_CSV_1)} and {os.path.join(output_folder_1, OUTPUT_EXCEL_1)}.")
    print(f"Bounding box log saved to {os.path.join(output_folder_1, OUTPUT_BBOX_CSV_1)}.")
    print(f"Logs saved for output_folder_2 to {os.path.join(output_folder_2, OUTPUT_CSV_2)} and {os.path.join(output_folder_2, OUTPUT_EXCEL_2)}.")
    print(f"Bounding box log saved to {os.path.join(output_folder_2, OUTPUT_BBOX_CSV_2)}.")

if __name__ == "__main__":
    main()


Successfully created new defected images.
Logs saved for output_folder_1 to /Users/saidev/Downloads/madara/1000/kim_1000.csv and /Users/saidev/Downloads/madara/1000/kim_1000.xlsx.
Bounding box log saved to /Users/saidev/Downloads/madara/1000/kim_bbox_1000.csv.
Logs saved for output_folder_2 to /Users/saidev/Downloads/madara/750/kim_750.csv and /Users/saidev/Downloads/madara/750/kim_750.xlsx.
Bounding box log saved to /Users/saidev/Downloads/madara/750/kim_bbox_750.csv.


In [87]:
import os
import re

def rename_images(directories):
    for directory in directories:
        for filename in os.listdir(directory):
            if os.path.isfile(os.path.join(directory, filename)):
                new_filename = clean_image_name(filename)
                old_filepath = os.path.join(directory, filename)
                new_filepath = os.path.join(directory, new_filename)
                os.rename(old_filepath, new_filepath)

def adjust_image_number(image_name):
    match = re.match(r'(\D+)(\d+)(\..+)', image_name)
    if match:
        prefix = match.group(1)
        number = int(match.group(2))
        suffix = match.group(3)
        new_number = number + 10000
        adjusted_name = f"{prefix}{new_number:05d}{suffix}"
        return adjusted_name
    return image_name

def clean_image_name(image_name):
    # Split the filename into parts separated by underscores, keeping the extension
    name_part, extension = os.path.splitext(image_name)
    parts = name_part.split('_')
    # Keep only the first two parts and add the original file extension
    cleaned_name = '_'.join(parts[:2]) + extension
    return cleaned_name

if __name__ == "__main__":
    image_directories = [
        '/Users/saidev/Downloads/madara/1000',
        '/Users/saidev/Downloads/madara/Json_1000',
]
    rename_images(image_directories)


In [89]:
import os
import json

def clean_image_name(image_name):
    # Split the filename into parts separated by underscores, keeping the extension
    name_part, extension = os.path.splitext(image_name)
    parts = name_part.split('_')
    # Keep only the first two parts and add the original file extension
    cleaned_name = '_'.join(parts[:2]) + extension
    return cleaned_name

def update_image_name_in_json_file(json_file_path):
    # Load the existing JSON data
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Update the image_name field
    if 'image_name' in data:
        old_image_name = data['image_name']
        data['image_name'] = clean_image_name(old_image_name)
    
    # Save the modified JSON data back to the file
    with open(json_file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def process_json_files_in_folders(json_folders):
    for json_folder in json_folders:
        json_files = [f for f in os.listdir(json_folder) if f.endswith('.json')]
        for json_file in json_files:
            json_path = os.path.join(json_folder, json_file)
            update_image_name_in_json_file(json_path)

if __name__ == "__main__":
    json_folders = [
        '/Users/saidev/Downloads/madara/Json_1000/',# Replace with the path to your first folder # Replace with the path to your second folder
    ]
    process_json_files_in_folders(json_folders)


In [91]:
import os
import csv

def clean_image_name(image_name):
    # Split the filename into parts separated by underscores, keeping the extension
    name_part, extension = os.path.splitext(image_name)
    parts = name_part.split('_')
    # Keep only the first two parts and add the original file extension
    cleaned_name = '_'.join(parts[:2]) + extension
    return cleaned_name

def update_image_name_in_csv_file(csv_file_path, column_name):
    # Load the existing CSV data
    with open(csv_file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = reader.fieldnames
    
    # Update the specified column in each row
    for row in rows:
        if column_name in row:
            old_image_name = row[column_name]
            row[column_name] = clean_image_name(old_image_name)
    
    # Save the modified CSV data back to the file
    with open(csv_file_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def process_csv_files(csv_files, column_name):
    for csv_file in csv_files:
        update_image_name_in_csv_file(csv_file, column_name)

if __name__ == "__main__":
    csv_files = [
        '/Users/saidev/Downloads/madara/1000/kim_1000.csv',
        '/Users/saidev/Downloads/madara/1000/kim_bbox.csv',
    ]
    column_name = 'image_name'  # Replace with the name of the column that contains the image names
    process_csv_files(csv_files, column_name)
